# Notebook 10 — Monitor and Improve Your Genie Space

**What you’ll learn:** How to track Genie’s accuracy over time, monitor user activity, identify questions that fail, and build a continuous improvement loop.

**Why this matters:** Deploying a Genie space is just the beginning. Over time, schemas change, new questions emerge, and models get updated. Monitoring ensures your space stays accurate and useful.

**What we’ll cover:**

| Part | What it shows | Data source |
|------|---------------|-------------|
| **1** | Genie’s built-in Monitoring tab | Genie UI |
| **2** | Benchmark pass rate over time | `genie_benchmark_runs` (from notebook 07) |
| **3** | User activity and feedback | `system.access.audit` |
| **4** | Query performance (latency) | `system.query.history` |
| **5** | Continuous improvement workflow | Best practices |

**Before you start:** Run notebooks **03** (spaces) and **07** (compare) for benchmark data.

**Compute:** Serverless. Some system table queries require elevated permissions.

## Configuration

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import html
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host_url = w.config.host.rstrip("/")

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host_url)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host_url}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
GENIE_SPACE_ID = PRIMARY_SPACE_ID
if not PRIMARY_SPACE_ID:
    print("WARNING: genie_space_id not found in workshop_config. Monitoring links will be incomplete. Run notebook 02 first.")



## Part 1: The Genie UI Monitoring Tab

The **built-in Monitoring tab** is your first stop. Open the Genie space and click the **Monitoring** tab (requires **CAN MANAGE** permission).

**What you can do there:**

- View **every question and answer** asked in the space
- Filter by **time range, user, rating, status**
- See **thumbs up/down** feedback from users
- Read **review requests** — users can flag a response for review and add a comment
- **Re-run query results** from other users' messages using your own credentials to debug
- **Comment on review requests** to confirm or correct responses

**Limitation:** Research Agent / Agent mode conversations currently appear greyed out in the Monitoring tab — you cannot click into those Q&A details yet.

In [ ]:
# Direct URL to the Genie Monitoring tab (Azure: /genie/rooms/<id>/monitoring?o=<workspace_id>)
m = re.search(r"adb-(\d+)\.", host_url)
o = f"?o={m.group(1)}" if m else ""
monitoring_url = f"{host_url}/genie/rooms/{GENIE_SPACE_ID}/monitoring{o}"
room_url = f"{host_url}/genie/rooms/{GENIE_SPACE_ID}{o}"
print("Monitoring (deep link):", monitoring_url)
print("Genie room (main UI):", room_url)
displayHTML(
    '<p><a href="'
    + html.escape(monitoring_url, quote=True)
    + '" target="_blank" rel="noopener">Open Genie Monitoring</a></p>'
    '<p style="color:#555;font-size:13px">If prompted, use <b>CAN MANAGE</b> on the space. '
    'Fallback: <a href="'
    + html.escape(room_url, quote=True)
    + '" target="_blank" rel="noopener">open the room</a> then choose the Monitoring tab.</p>'
)


## Part 2: Benchmark run history

Benchmark pass rate is the **most objective** measure of Genie accuracy. This reads the `genie_benchmark_runs` table written by notebook **07** — track pass rate over time as you tune instructions and curated SQL.

### 2a. Pass rate by run date

In [ ]:
try:
    display(spark.sql(f"""
        SELECT
            DATE(run_timestamp) AS run_date,
            COUNT(*) AS total_questions,
            SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed,
            ROUND(SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pass_rate_pct
        FROM {fqn}.genie_benchmark_runs
        GROUP BY DATE(run_timestamp)
        ORDER BY run_date DESC
    """))
except Exception as e:
    print(f"Run notebook 06 first to create genie_benchmark_runs: {str(e)[:150]}")

### 2b. Which questions fail most often?

In [ ]:
try:
    display(spark.sql(f"""
        SELECT
            benchmark_id,
            question,
            COUNT(*) AS runs,
            SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passes,
            SUM(CASE WHEN status != 'PASS' THEN 1 ELSE 0 END) AS non_passes,
            ROUND(SUM(CASE WHEN status != 'PASS' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS non_pass_rate_pct
        FROM {fqn}.genie_benchmark_runs
        GROUP BY benchmark_id, question
        ORDER BY non_pass_rate_pct DESC
    """))
except Exception as e:
    print(f"Table not available: {str(e)[:150]}")


## Part 3: Usage monitoring from audit logs

The `system.access.audit` table captures every Genie interaction under `service_name = 'aibiGenie'`.

**Key audit action names:**

| Action | Meaning |
|--------|---------|
| `genieStartConversationMessage` | New conversation started |
| `genieCreateConversationMessage` | Follow-up message in a conversation |
| `updateConversationMessageFeedback` | Thumbs up or thumbs down |
| `createConversationMessageComment` | Review request submitted |
| `genieExecuteMessageQuery` | SQL query executed by Genie |

**Note:** These queries require access to `system.access.audit`. If you get a permissions error, ask your metastore admin for `SELECT` access or skip to Part 5.

### 3a. Active users per day

In [ ]:
try:
    display(spark.sql("""
        SELECT
            event_date,
            COUNT(DISTINCT user_identity.email) AS active_users,
            COUNT(*) AS total_messages
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name IN ('genieStartConversationMessage', 'genieCreateConversationMessage')
          AND event_date >= current_date() - INTERVAL 30 DAYS
        GROUP BY event_date
        ORDER BY event_date
    """))
except Exception as e:
    print(f"system.access.audit not available: {str(e)[:200]}")
    print("Ask your metastore admin for SELECT access, or skip to Part 5.")

### 3b. Feedback ratings (thumbs up / thumbs down)

In [ ]:
try:
    display(spark.sql("""
        SELECT
            event_date,
            user_identity.email AS user_email,
            request_params.space_id,
            request_params.feedback_rating
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name = 'updateConversationMessageFeedback'
          AND event_date >= current_date() - INTERVAL 30 DAYS
        ORDER BY event_date DESC
    """))
except Exception as e:
    print(f"Not available: {str(e)[:200]}")

### 3c. Weekly approval rate trend

In [ ]:
try:
    display(spark.sql("""
        SELECT
            DATE_TRUNC('week', event_date) AS week,
            SUM(CASE WHEN request_params.feedback_rating = 'THUMBS_UP' THEN 1 ELSE 0 END) AS thumbs_up,
            SUM(CASE WHEN request_params.feedback_rating = 'THUMBS_DOWN' THEN 1 ELSE 0 END) AS thumbs_down,
            COUNT(*) AS total_ratings,
            ROUND(
                SUM(CASE WHEN request_params.feedback_rating = 'THUMBS_UP' THEN 1 ELSE 0 END) * 100.0
                / NULLIF(COUNT(*), 0), 1
            ) AS approval_rate_pct
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name = 'updateConversationMessageFeedback'
          AND event_date >= current_date() - INTERVAL 90 DAYS
        GROUP BY DATE_TRUNC('week', event_date)
        ORDER BY week
    """))
except Exception as e:
    print(f"Not available: {str(e)[:200]}")

### 3d. Review requests

In [ ]:
try:
    display(spark.sql("""
        SELECT
            event_date,
            user_identity.email AS reviewer,
            request_params.space_id,
            action_name
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name = 'createConversationMessageComment'
          AND event_date >= current_date() - INTERVAL 30 DAYS
        ORDER BY event_date DESC
    """))
except Exception as e:
    print(f"Not available: {str(e)[:200]}")

### 3e. API error tracking (last 7 days)

In [ ]:
try:
    display(spark.sql("""
        SELECT
            DATE_TRUNC('hour', event_time) AS hour,
            response.status_code,
            COUNT(*) AS request_count
        FROM system.access.audit
        WHERE service_name = 'aibiGenie'
          AND action_name IN ('genieStartConversationMessage', 'genieCreateConversationMessage')
          AND event_date >= current_date() - INTERVAL 7 DAYS
        GROUP BY DATE_TRUNC('hour', event_time), response.status_code
        ORDER BY hour DESC, response.status_code
    """))
except Exception as e:
    print(f"Not available: {str(e)[:200]}")


## Part 4: Query performance from query history

`system.query.history` captures execution details for Genie-originated SQL. Filter with `client_application = 'Databricks SQL Genie Space'`.

### 4a. Genie query latency (p50, p95)

In [ ]:
try:
    display(spark.sql("""
        SELECT
            DATE(start_time) AS query_date,
            COUNT(*) AS genie_queries,
            ROUND(percentile_approx(total_duration_ms, 0.5) / 1000.0, 2) AS p50_seconds,
            ROUND(percentile_approx(total_duration_ms, 0.95) / 1000.0, 2) AS p95_seconds,
            ROUND(AVG(total_duration_ms) / 1000.0, 2) AS avg_seconds
        FROM system.query.history
        WHERE client_application = 'Databricks SQL Genie Space'
          AND start_time >= current_date() - INTERVAL 30 DAYS
        GROUP BY DATE(start_time)
        ORDER BY query_date DESC
    """))
except Exception as e:
    print(f"system.query.history not available: {str(e)[:200]}")

### 4b. Recent Genie-executed SQL statements

In [ ]:
try:
    display(spark.sql("""
        SELECT
            start_time,
            executed_by,
            SUBSTRING(statement_text, 1, 200) AS sql_preview,
            total_duration_ms,
            produced_rows
        FROM system.query.history
        WHERE client_application = 'Databricks SQL Genie Space'
          AND start_time >= current_date() - INTERVAL 7 DAYS
        ORDER BY start_time DESC
        LIMIT 25
    """))
except Exception as e:
    print(f"Not available: {str(e)[:200]}")


## Part 5: The continuous improvement loop

Monitoring is only useful if it drives action. Here is the recommended workflow for iterating on a Genie space after deployment.

### Step 1: Triage feedback

1. Open the **Monitoring tab** in the Genie UI.
2. Filter for **thumbs down** and **review requests** from the last week.
3. For each thumbs-down response:
   - Click to see the **question asked** and the **SQL Genie generated**.
   - Re-run the query with your credentials to verify the result.
   - Categorize the failure: **wrong table**, **wrong join**, **wrong metric formula**, **wrong filter**, or **hallucinated column**.

### Step 2: Follow the fix order

Fix in this order — each level resolves entire classes of downstream failures:

| Priority | Fix | Example |
|----------|-----|---------|
| **1. Data / views** | Create a view or metric view that pre-computes the right grain | Add `oee_monthly_summary` view |
| **2. Metadata** | Add/improve table and column comments in Unity Catalog | Comment `oee_score` as "0-1 scale, multiply by 100 for %" |
| **3. Joins** | Document missing or ambiguous join paths in instructions | "Join safety_incidents to plants through production_lines" |
| **4. Example SQL** | Add curated Q-to-SQL pairs for the failing question pattern | Add "defect rate by shift" example |
| **5. Instructions** | Update text instructions for edge cases or thresholds | Add scrap rate classification bands |

### Step 3: Re-run benchmarks

After each change, re-run notebook **07** to measure the impact. Track pass rate over time — you should see monotonic improvement.

### Step 4: Convert good interactions to curated SQL

When users ask great questions that Genie answers correctly, **promote** them:
1. Copy the question and SQL from the Monitoring tab.
2. Add as a new `example_question_sql` entry in the space (via notebook 02 rebuild or PATCH API from notebook 08).
3. This teaches Genie to handle similar future questions reliably.

### Step 5: Schedule automated benchmarks

For production spaces, schedule notebook **07** as a recurring Databricks Job (weekly or after instruction changes). Alert on pass rate drops.


## Part 6: Community tools for Genie space health

These open-source tools automate parts of the improvement loop:

| Tool | What it does | Link |
|------|-------------|------|
| **GenieRx** | Scores your space config against best practices; suggests optimizations | [github.com/hiydavid/dbx-genie-rx](https://github.com/hiydavid/dbx-genie-rx) |
| **GenieIQ** | Scores spaces 0–100, generates "fix notebooks," tracks progress in Lakebase | [github.com/ryancicak/genieiq](https://github.com/ryancicak/genieiq) |
| **Databricks Forge** | Quality / maturity grades A–F across 40 checks; auto-improve loop | [althrussell.github.io/databricks-forge](https://althrussell.github.io/databricks-forge/) |

### Recommended reading

- [How to Build Production-Ready Genie Spaces](https://www.databricks.com/blog/how-build-production-ready-genie-spaces-and-build-trust-along-way) — benchmark-driven curation guide 
- [Curate an effective Genie space](https://docs.databricks.com/aws/en/genie/best-practices) — official best practices
- [Build a knowledge store for Genie](https://docs.databricks.com/aws/en/genie/knowledge-store) — external knowledge integration
- [Monitor Genie with audit logs](https://docs.databricks.com/aws/en/ai-bi/admin/audit) — official audit log reference


## Privacy and compliance note

When building monitoring dashboards from audit and query history:

- **Do not expose PII** — aggregate metrics only. Do not surface individual user emails in shared dashboards without HR/legal approval.
- **Query text may contain sensitive data** — `system.query.history.statement_text` can include filter values from user questions. Restrict dashboard access accordingly.
- **Retention** — system tables have workspace-defined retention periods. For long-term trend tracking, materialize aggregated metrics into your own Delta tables.